# GFW threshold matching to ASTD *a.k.a Labeling ASTD*
The notebook goes through the GFW matching of fishing vessels on ASTD fishing vessels \
We compare the GFW to the ASTD fishing vessels paths for each month to find associated mmsi - shipid

The workflow is further explained and detailed in [GFW2ASTD.md](GFW2ASTD.md) file

## The comparison is done with data aggregated day by day, over multiple years

In [1]:
from backend import *
%load_ext autoreload
%autoreload 2

gfw_path = "D:/Stockage/GFW/"
astd_path = r"D:\Stockage\ASTD"
parquet_path = "../../examples/data/"

months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
lat_threshold = 59

# Periods for load_periods
periods = {
    2016: months,
    2017: months,
    2018: months,
    2019: months,
    2020: months,
}

In [2]:
## To reduce compute overhead, we pre-preprocess the data so that we have a lighter version to complete the matchings
## For astd, we choose only fishing vessels which greatly reduce it's size, and for GFW we aggregate every files (per year) with records per day and positions
df_ASTD = load_periods('all_fishing_2016-2020.parquet', source=astd_path, periods= periods, remove_nan_rows="default", usecols="default", progress=True)
df_ASTD["date_time_utc"] = pd.to_datetime(df_ASTD["date_time_utc"])

display(df_ASTD.sample(1))

## GFW aggregation
# agg = []
# for file in os.listdir(parquet_path):
#     if file.startswith("gfw_20") and file.endswith(".parquet"):
#         agg.append(file)

# conc = []

# for parquet_file in agg:
#     period = pd.read_parquet(parquet_path + parquet_file)
#     print(parquet_file)
#     period['date'] = pd.to_datetime(period['date'])
#     period['month'] = period['date'].dt.to_period('M')

#     # Converting grid origin to integer (this avoid float issues with imprecisions)
#     period["grid_lon"] = (period["cell_ll_lon"] * 10).astype('int16')
#     period["grid_lat"] = (period["cell_ll_lat"] * 10).astype('int16')

#     # for col in ["hours", "fishing_hours"]:
#     #     period[col] = pd.to_numeric(period[col], downcast="float")

#     period.drop(columns=['hours', 'fishing_hours', 'cell_ll_lat', 'cell_ll_lon'], inplace=True)
#     period = period.drop_duplicates(ignore_index=True).copy()

#     conc.append(period)

# the_end = pd.concat(conc, ignore_index=True)
# the_end.to_parquet(parquet_path + "GFWLIGHT_2016-2020.parquet", index=False)

df_GFW = load_periods_gfw('GFWLIGHT_2016-2020.parquet', source=gfw_path, periods = periods)

display(df_GFW.sample(1))

Loaded period ../../examples/data/all_fishing_2016-2020.parquet, parameters ignored


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
70048519,2214,2020-11-11 02:47:32+00:00,Iceland,FS Ice Class 1C,Fishing vessels,< 1000 GT,3.069,370,-21.943014,64.154465


Loaded data ../../examples/data/GFWLIGHT_2016-2020.parquet, parameters ignored


,date,mmsi,month,grid_lon,grid_lat
58814740,2017-08-17,227984000,2017-08,-82,443


## Process GFW
### Basic processing

In [3]:
### Basic GFW pre-processing on raw data after basic import (not light data)

# Define starting and ending time for the grid (max a month)
# df_GFW['month'] = df_GFW['date'].dt.year.astype(str) + '-' + df_GFW['date'].dt.month.astype(str).str.zfill(2)

# df_GFW['date'] = pd.to_datetime(df_GFW['date'])
# df_GFW['month'] = df_GFW['date'].dt.to_period('M')

# # Converting grid origin to integer (this avoid float issues with imprecisions)
# df_GFW["grid_lon"] = (df_GFW["cell_ll_lon"] * 10).astype('int16')
# df_GFW["grid_lat"] = (df_GFW["cell_ll_lat"] * 10).astype('int16')

# for col in ["hours", "fishing_hours"]:
#     df_GFW[col] = pd.to_numeric(df_GFW[col], downcast="float")

# df_GFW.drop(columns=['hours', 'fishing_hours', 'cell_ll_lat', 'cell_ll_lon'], inplace=True)


### Filter lat and positions

In [4]:
# Select mmsi with more than a month of data
counts = df_GFW.groupby('mmsi')['month'].nunique()
valid_mmsi = counts[counts > 1].index

gfw_work = df_GFW[df_GFW['mmsi'].isin(valid_mmsi)].copy()

del counts, valid_mmsi

In [5]:
# Filter remove all mmsi with points under europe line - lat = 59:
gfw_work = filter_latlon(gfw_work, lat=lat_threshold)

# # Original aggregation of the data by day and by positions (after latitude filter, if not already done) - don't run because we might loose
# gfw_work = gfw_work[['mmsi', 'date', 'grid_lat', 'grid_lon', 'month']].drop_duplicates()
# gfw_work

## Process ASTD

### Basic processing

In [6]:
# df_ASTD['month'] = df_ASTD['date_time_utc'].dt.year.astype(str) + '-' + df_ASTD['date_time_utc'].dt.month.astype(str).str.zfill(2)
dt = df_ASTD['date_time_utc'].dt
df_ASTD['month'] = dt.to_period('M')
df_ASTD['date'] = dt.floor('D')

del dt

# Transform ASTD into grids to reduce compute overhead
df_ASTD["grid_lon"] = np.floor(df_ASTD["longitude"] * 10).astype('int16')
df_ASTD["grid_lat"] = np.floor(df_ASTD["latitude"] * 10).astype('int16')

df_ASTD.sample(1)

C:\Users\virtu\AppData\Local\Temp\ipykernel_17376\3630638983.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_ASTD['month'] = dt.to_period('M')


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,date,grid_lon,grid_lat
18633565,1891,2017-06-06 21:45:35+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,12.208402,361,18.997547,69.679863,2017-06,2017-06-06 00:00:00+00:00,189,696


In [7]:
# Aggregating the data by day and by positions
df_astd = df_ASTD[['date', 'shipid', 'grid_lat', 'grid_lon', 'month', 'iceclass', 'sizegroup_gt', 'flagname']].drop_duplicates()
display(df_astd.sample(1))

,date,shipid,grid_lat,grid_lon,month,iceclass,sizegroup_gt,flagname
30333801,2018-03-24 00:00:00+00:00,2393,707,299,2018-03,FS Ice Class 1C,1000 - 4999 GT,Faeroe Islands


In [9]:
# Filter remove all points under europe line - lat = 59:
df_astd = filter_latlon(df_astd, lat=lat_threshold, gfw=False)

## Merge (inner join) both by position and time

In [10]:
gfw_work['date'] = pd.to_datetime(gfw_work['date'], utc=True)
df_astd['date'] = pd.to_datetime(df_astd['date'], utc=True)

merged = gfw_work.merge(df_astd, on=["grid_lon", "grid_lat", "date", "month"], how="inner")
merged

,date,mmsi,month,grid_lon,grid_lat,shipid,iceclass,sizegroup_gt,flagname
0,2016-01-01 00:00:00+00:00,231046000,2016-01,-62,612,3967,FS Ice Class 1C,< 1000 GT,Faeroe Islands
1,2016-01-01 00:00:00+00:00,231046000,2016-01,-63,613,3967,FS Ice Class 1C,< 1000 GT,Faeroe Islands
2,2016-01-01 00:00:00+00:00,231046000,2016-01,-64,613,3967,FS Ice Class 1C,< 1000 GT,Faeroe Islands
3,2016-01-01 00:00:00+00:00,231046000,2016-01,-64,613,3980,FS Ice Class 1C,1000 - 4999 GT,Faeroe Islands
4,2016-01-01 00:00:00+00:00,231046000,2016-01,-66,614,3967,FS Ice Class 1C,< 1000 GT,Faeroe Islands
...,...,...,...,...,...,...,...,...,...
9604785,2020-12-31 00:00:00+00:00,273352280,2020-12,87,792,1634,FS Ice Class 1B,1000 - 4999 GT,Russia
9604786,2020-12-31 00:00:00+00:00,273352280,2020-12,87,792,1772,FS Ice Class 1B,1000 - 4999 GT,Russia
9604787,2020-12-31 00:00:00+00:00,273352280,2020-12,86,793,1772,FS Ice Class 1B,1000 - 4999 GT,Russia
9604788,2020-12-31 00:00:00+00:00,273352280,2020-12,84,793,1772,FS Ice Class 1B,1000 - 4999 GT,Russia


## Get shipids per mmsi

In [11]:
# Number of matched points per shipid, per mmsi, per month
match_size = (
    merged
    .groupby(['mmsi', 'month', 'shipid'])
    .agg(match_n_mmsi=('mmsi', 'size'))
    .reset_index()
)

# Total ASTD number of points per shipid, per month
astd_total = (
    df_astd
    .groupby(['shipid', 'month'])
    .agg(astd_n_ship=('shipid', 'size'),
        iceclass=('iceclass', 'first'),
        sizegroup_gt=('sizegroup_gt', 'first'),
        flagname=('flagname', 'first'))
    .reset_index()
)

# Total selected GFW number of points per mmsi, per month
gfw_total = (
    gfw_work
    .groupby(['mmsi', 'month'])
    .agg(gfw_n_mmsi=('mmsi', 'size'))
    .reset_index()
)

# Merge all
merged_scoring = (
    match_size
    .merge(astd_total, on=['shipid', 'month'], how='inner')
    .merge(gfw_total, on=['mmsi', 'month'], how='inner')
)

# Compute scores
merged_scoring['ratio_gfw'] = merged_scoring['match_n_mmsi'] / merged_scoring['gfw_n_mmsi'] # Proportion of matched gfw points
merged_scoring['ratio_astd'] = merged_scoring['match_n_mmsi'] / merged_scoring['astd_n_ship'] # Proportion of matched astd points

merged_scoring['score'] = merged_scoring['ratio_gfw'] + merged_scoring['ratio_astd']
merged_scoring['max_score_mmsi'] = merged_scoring.groupby(['mmsi', 'month', 'iceclass', 'sizegroup_gt', 'flagname'])['score'].transform('max')
merged_scoring['max_score_shipid'] = merged_scoring.groupby(['shipid', 'month'])['score'].transform('max')

merged_scoring_high = merged_scoring[
    ## Takes highest mmsi for a shipid, that's also the highest shipid for a mmsi
    (merged_scoring['score'] == merged_scoring['max_score_mmsi'])
    & (merged_scoring['score'] == merged_scoring['max_score_shipid'])
]

merged_scoring_high.to_csv(parquet_path + f"gfw_threshmatch2016_2020_HIGHESTLOGS.{lat_threshold}.csv", index=False)
merged_scoring_high

C:\Users\virtu\AppData\Local\Temp\ipykernel_17376\3013397867.py:40: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged_scoring['max_score_mmsi'] = merged_scoring.groupby(['mmsi', 'month', 'iceclass', 'sizegroup_gt', 'flagname'])['score'].transform('max')


,mmsi,month,shipid,match_n_mmsi,astd_n_ship,iceclass,sizegroup_gt,flagname,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,max_score_shipid
76,57084140,2017-05,8309,4,18,FS Ice Class 1C,1000 - 4999 GT,Norway,75,0.053333,0.222222,0.275556,0.275556,0.275556
330,57173720,2017-08,2996,12,118,FS Ice Class 1C,1000 - 4999 GT,Norway,25,0.480000,0.101695,0.581695,0.581695,0.581695
350,230940400,2016-06,3835,2,68,FS Ice Class 1C,1000 - 4999 GT,Sweden,122,0.016393,0.029412,0.045805,0.045805,0.045805
355,230987140,2016-01,6613,12,314,FS Ice Class 1C,< 1000 GT,Sweden,142,0.084507,0.038217,0.122724,0.122724,0.122724
357,230987140,2019-01,3317,2,312,FS Ice Class 1C,< 1000 GT,Sweden,134,0.014925,0.006410,0.021336,0.021336,0.021336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972758,331781000,2020-02,3201,97,451,FS Ice Class 1B,1000 - 4999 GT,Denmark,386,0.251295,0.215078,0.466373,0.466373,0.466373
972766,331781000,2020-03,3772,7,274,FS Ice Class 1C,1000 - 4999 GT,Denmark,20,0.350000,0.025547,0.375547,0.375547,0.375547
972774,331784000,2020-08,2646,3,788,FS Ice Class 1C,1000 - 4999 GT,Canada,73,0.041096,0.003807,0.044903,0.044903,0.044903
972798,331784000,2020-11,2713,4,542,FS Ice Class 1B,1000 - 4999 GT,Canada,131,0.030534,0.007380,0.037914,0.037914,0.037914


### Compute thresholds

In [12]:
# Thresholds (set to 1 if a strict match is needed)
t_astd = 0.9
t_gfw_high = 0.6
t_gfw_low = 0.8
n_small = 20

In [13]:
mask_merged = (
    ( # = allowed missing astd datapoints
        ((merged_scoring_high['match_n_mmsi'] < n_small) & (merged_scoring_high['ratio_gfw'] >= t_gfw_low)) | # Ratio has to be t_gfw_low with tracks with less than n_small points
        ((merged_scoring_high['match_n_mmsi'] >= n_small) & (merged_scoring_high['ratio_gfw'] >= t_gfw_high)) # Else, check if ratio is higher than t_gfw_high
    )
    # = allowed missing gfw datapoints
    & (merged_scoring_high['ratio_astd'] >= t_astd) # Minimum expected ratio of ASTD found in GFW (e.g. 90% of ASTD has to be found in GFW to be considered)
)

merged_scores = merged_scoring_high[mask_merged].copy()
merged_scores = merged_scores.sort_values(by='gfw_n_mmsi', ascending=False).reset_index(drop=True)


merged_scores.to_csv(parquet_path + f"gfw_threshmatch2016_2020_LOGS.{lat_threshold}{[t_astd,t_gfw_low,t_gfw_high,n_small]}.csv", index=False)

merged_scores

,mmsi,month,shipid,match_n_mmsi,astd_n_ship,iceclass,sizegroup_gt,flagname,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,max_score_shipid
0,273451570,2020-09,324,1274,1290,FS Ice Class 1C,< 1000 GT,Russia,1556,0.818766,0.987597,1.806363,1.806363,1.806363
1,258535000,2020-10,1219,1301,1370,FS Ice Class 1B,1000 - 4999 GT,Norway,1513,0.859881,0.949635,1.809516,1.809516,1.809516
2,251153000,2019-06,2031,1093,1145,FS Ice Class 1C,1000 - 4999 GT,Iceland,1490,0.733557,0.954585,1.688142,1.688142,1.688142
3,251318000,2019-06,2591,1085,1139,FS Ice Class 1C,1000 - 4999 GT,Iceland,1436,0.755571,0.952590,1.708161,1.708161,1.708161
4,258535000,2019-06,3657,1153,1170,FS Ice Class 1B,1000 - 4999 GT,Norway,1398,0.824750,0.985470,1.810220,1.810220,1.810220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4638,257046460,2020-10,478,2,2,FS Ice Class 1A,1000 - 4999 GT,Norway,2,1.000000,1.000000,2.000000,2.000000,2.000000
4639,231189000,2020-05,22009,2,2,FS Ice Class 1C,< 1000 GT,Faeroe Islands,2,1.000000,1.000000,2.000000,2.000000,2.000000
4640,258205000,2018-07,7890,2,2,FS Ice Class 1C,< 1000 GT,Norway,2,1.000000,1.000000,2.000000,2.000000,2.000000
4641,277558000,2019-05,15527,2,2,FS Ice Class 1B,1000 - 4999 GT,Lithuania,2,1.000000,1.000000,2.000000,2.000000,2.000000


### Compute class verification for final labels

In [14]:
df = merged_scores.copy()

# # Remove multiple shipid for one mmsi (in case many have the same max score)
# mask = df.groupby(['mmsi', 'month', 'iceclass', 'sizegroup_gt', 'flagname'])['shipid'].transform('nunique') == 1

# # Remove multiple mmsi for one shipid
# mask1 = df.groupby(['shipid', 'month'])['mmsi'].transform('nunique') == 1

# df = df[mask1 & mask]
# df = df.sort_values(by='match_n_mmsi', ascending=False).reset_index(drop=True)

# Keep only the same class for each mmsi (the one that comes more often)
cols = ['iceclass', 'sizegroup_gt', 'flagname']
mode_df = (
    df.groupby(['mmsi'])[cols]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
      .reset_index()
)

df = df.merge(
    mode_df,
    on=['mmsi'],
    suffixes=('', '_mode')
)

# Keep shipid with the same classes
df = df[
    (df['iceclass'] == df['iceclass_mode']) &
    (df['sizegroup_gt'] == df['sizegroup_gt_mode']) &
    (df['flagname'] == df['flagname_mode'])
]

df.drop(columns=['iceclass_mode', 'sizegroup_gt_mode', 'flagname_mode'], inplace=True)
df


,mmsi,month,shipid,match_n_mmsi,astd_n_ship,iceclass,sizegroup_gt,flagname,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,max_score_shipid
0,273451570,2020-09,324,1274,1290,FS Ice Class 1C,< 1000 GT,Russia,1556,0.818766,0.987597,1.806363,1.806363,1.806363
1,258535000,2020-10,1219,1301,1370,FS Ice Class 1B,1000 - 4999 GT,Norway,1513,0.859881,0.949635,1.809516,1.809516,1.809516
2,251153000,2019-06,2031,1093,1145,FS Ice Class 1C,1000 - 4999 GT,Iceland,1490,0.733557,0.954585,1.688142,1.688142,1.688142
3,251318000,2019-06,2591,1085,1139,FS Ice Class 1C,1000 - 4999 GT,Iceland,1436,0.755571,0.952590,1.708161,1.708161,1.708161
4,258535000,2019-06,3657,1153,1170,FS Ice Class 1B,1000 - 4999 GT,Norway,1398,0.824750,0.985470,1.810220,1.810220,1.810220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4638,257046460,2020-10,478,2,2,FS Ice Class 1A,1000 - 4999 GT,Norway,2,1.000000,1.000000,2.000000,2.000000,2.000000
4639,231189000,2020-05,22009,2,2,FS Ice Class 1C,< 1000 GT,Faeroe Islands,2,1.000000,1.000000,2.000000,2.000000,2.000000
4640,258205000,2018-07,7890,2,2,FS Ice Class 1C,< 1000 GT,Norway,2,1.000000,1.000000,2.000000,2.000000,2.000000
4641,277558000,2019-05,15527,2,2,FS Ice Class 1B,1000 - 4999 GT,Lithuania,2,1.000000,1.000000,2.000000,2.000000,2.000000


In [15]:
# Save into csv
df.to_csv(parquet_path + f"gfw_threshmatch2016_2020.{lat_threshold}{[t_astd,t_gfw_low,t_gfw_high,n_small]}.csv", index=False)

## Display examples

In [ ]:
mmsi = 257032830

display(gfw_work[gfw_work['mmsi']==mmsi])

In [ ]:
# Convert to a dictionnary - makes it easier to get the shipid for each mmsi
result = {}

for _, row in df.iterrows():
    mmsi = row['mmsi']
    month = row['month']
    shipid = row['shipid']
    result.setdefault(mmsi, {})[month] = shipid


top_ships = result[mmsi]
# mmsi=273418680
# top_ships = {'2020-02' : 2915}

top_ships_df = pd.DataFrame(list(top_ships.items()), columns=['month', 'shipid'])

col_lat_astd = 'grid_lat'
col_lon_astd = 'grid_lon'

col_lat_gfw = 'grid_lat'
col_lon_gfw = 'grid_lon'

# To plot astd origin point or grids + gfw sample or not
origin = True

if origin:

    col_lat_astd = 'latitude'
    col_lon_astd = 'longitude'

    # col_lat_gfw = 'cell_ll_lat'
    # col_lon_gfw = 'cell_ll_lon'
    df1 = df_ASTD.merge(top_ships_df, on=['shipid', 'month'], how='inner')
    df1['track_id'] = mmsi
    df2 = df_GFW[df_GFW['mmsi']==mmsi].merge(top_ships_df, on=['month'], how='inner')

else:
    df2 = gfw_work[(gfw_work['mmsi']==mmsi)].merge(top_ships_df, on=['month'], how='inner')
    df1 = df_astd.merge(top_ships_df, on=['shipid', 'month'], how='inner')

df2 = df2.drop_duplicates(subset=[col_lon_gfw, col_lat_gfw])

display(df2)
df1

In [ ]:
res = 0.1

pio.renderers.default = "browser"
# pio.renderers.default = "notebook"

# Plot ASTD
if origin:
    fig = tb.plot_ship_tracks(
        df1,
        track_ids=[mmsi],
        color_by="track_id",
        show_points=True,
        show_start_end=True,
        map_style="open-street-map",
        zoom=4,
    )

else:
    fig = go.Figure()
    lons_all = []
    lats_all = []

    for _, row in df1.iterrows():
        lon0 = row[col_lon_astd] * res
        lat0 = row[col_lat_astd] * res

        lons_all += [lon0, lon0+0.1, lon0+0.1, lon0, lon0, None]
        lats_all += [lat0, lat0, lat0+0.1, lat0+0.1, lat0, None]

    fig.add_trace(go.Scattermap(
        lon=lons_all,
        lat=lats_all,
        mode="lines",
        line=dict(color="blue", width=2),
        name="Cells df1"
    ))

#GFW plot grids
lons_all = []
lats_all = []

for _, row in df2.iterrows():
    lon0 = row[col_lon_gfw] * res
    lat0 = row[col_lat_gfw] * res

    lons_all += [lon0, lon0+0.1, lon0+0.1, lon0, lon0, None]
    lats_all += [lat0, lat0, lat0+0.1, lat0+0.1, lat0, None]


fig.add_trace(go.Scattermap(
    lon=lons_all,
    lat=lats_all,
    mode="lines",
    line=dict(color="red", width=1),
    name="Cells df2"
))

fig.update_layout(
    map=dict(
        style="open-street-map",
        zoom=3,
        center=dict(lat=df1[col_lat_astd].mean(),
                    lon=df1[col_lon_astd].mean())
    ),
    title=f"{mmsi} for top shipids"
)

# tb.export_figure(fig, f"GFW2ASTD_{mmsi}.html")

fig.show()

## Global overview of match, gfw, and astd datapoints on a map


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

t1 = gfw_work[gfw_work['month'] == "2017-02"]
t2 = df_astd[df_astd['month'] == "2017-02"]

t_merge = merged[merged['month'] == "2017-02"]

fig.add_trace(go.Scattermap(
    lon=t1["grid_lon"] / 10,
    lat=t1["grid_lat"] / 10,
    mode="markers",
    marker=dict(size=5, color="blue"),
    name="GFW"
))

# ASTD points
fig.add_trace(go.Scattermap(
    lon=t2["grid_lon"] / 10,
    lat=t2["grid_lat"] / 10,
    mode="markers",
    marker=dict(size=5, color="green"),
    name="ASTD"
))

# Merged points
fig.add_trace(go.Scattermap(
    lon=t_merge["grid_lon"] / 10,
    lat=t_merge["grid_lat"] / 10,
    mode="markers",
    marker=dict(size=5, color="red"),
    name="Merged"
))
pio.renderers.default = "browser"
fig.show()